Debugging mindset in RAG
If a RAG system gives an incorrect answer, do not immediately assume the LLM is the problem.

Investigate in order:
1. Retrieval: Did the system find the correct document?
2. Context construction: Did the prompt include the relevant passages without truncating important information?
3. Generation: Did the LLM correctly interpret the supplied evidence?
4. Data quality: Was the source accurate and up to date?
5. Evaluation: Can you reproduce the failure with a test question?
This diagnostic approach will be useful throughout the course.

Imagine a knowledge assistant retrieves an unrelated document. Which component failed first, and what would you investigate?

In your own words, why can RAG still produce a wrong answer even when it retrieves documents?

A new policy was uploaded, but the assistant still retrieves the old policy. Explain how you would investigate and resolve the issue.

What are RAG's main limitations?
- Poor retrieval: The system may fail to find the correct evidence.
- Stale data: The index may not reflect recent document updates.
- Hallucinations: The LLM may add unsupported claims.
- Context limits: Too much retrieved information can exceed the model's context budget.
- Latency and cost: Embedding queries, searching, reranking, and calling an LLM all add overhead.
- Security risks: A system can expose restricted information if authorization and retrieval controls are inadequate.

A RAG application retrieves ten long documents for a simple question, causing high latency and cost. What could you improve?

• First, I would optimize the Chunking Strategy. If the system is pulling 'long documents,' the chunk sizes are likely too large. I would break the text down into smaller, bite-sized fragments—around 100 to 250 tokens each. This ensures the system retrieves only the exact paragraph containing the answer, not the entire chapter.
• Second, I would tighten the Retrieval Constraints. I'd lower the Top-K parameter from 10 down to 3 or 5, as a simple question rarely requires ten sources. I would also introduce a Similarity Score Threshold so the system only fetches documents that closely match the query, rather than forcing itself to return a fixed count.
• Third, I would introduce a Reranking Layer. I’d place a lightweight reranker (like Cohere or BGE) between the vector database and the LLM. The vector database can broadly pull the top 10 chunks, but the reranker will evaluate their exact relevance, filter out the noise, and send only the top 2 or 3 definitive chunks to the LLM.
By shrinking the chunks, lowering the retrieval count, and filtering with a reranker, we sharply reduce the input tokens. This slashes API costs, minimizes latency, and keeps the LLM focused on a clean, concise context."

. A company wants an assistant to answer questions using frequently updated internal manuals. Why might RAG be a good fit?

4. Why not embed an entire document?
There are two separate problems to consider.

Problem A — Embedding limits. Embedding models have input limits, and a very large document may need to be split before it can be embedded.

Problem B — Retrieval precision. Even if the whole document fits within the embedding model's limits, its vector represents a broad mixture of topics. A smaller, focused chunk can make it easier to retrieve the specific information relevant to a question.

Chunking therefore affects how well the retrieval system can locate evidence, not just whether the document fits into a model.

**Module 2 Begins here  Document processing and chunking**

What is recursive character splitting?


Intuition: Instead of cutting text at arbitrary positions immediately, try to split it at natural boundaries first.

A recursive character splitter typically tries separators in this order:
1. "\n\n" — paragraph boundaries.
2. "\n" — line boundaries.
3. " " — word boundaries.
4. "" — individual characters, as a last resort.
If a paragraph is too large, the splitter moves to a finer separator. This continues until the resulting pieces fit the configured chunk size, subject to the splitting algorithm's rules.

in what cases this above chunking strategy is best ?

Markdown-aware chunking 



Semantic chunking

Semantic chunking tries to split text where the meaning changes, rather than relying only on character counts or formatting.

Industry recommendation: Start with recursive splitting as your baseline. Evaluate retrieval quality on representative queries, then test semantic chunking if documents contain frequent topic changes that structural splitting cannot handle well. Choose based on measured results, not because a more advanced technique sounds better.

Parent-child and hierarchical chunking

Parent-child chunking uses smaller child chunks for precise retrieval and their associated larger parent chunks to provide broader context to the LLM.

The child chunks are indexed for retrieval. Each child stores a reference to its parent. When a child matches the user's query, the system can retrieve the associated parent section to provide broader context to the LLM.

A typical implementation has these components:
- Child index: Stores embeddings of small chunks and their identifiers.
- Parent store: Stores larger text sections, often keyed by parent ID.
- Reference mapping: Connects each child to its parent.
- Context builder: Fetches parent content and decides how much to pass to the LLM.

5. When should you use parent-child chunking?
It is particularly useful when:
- Your documents contain long procedures with multiple dependent steps.
- A precise passage is easy to retrieve, but insufficient to answer safely or correctly.
- You need to preserve prerequisites, exceptions, or supporting conditions.
- Standard chunking performs poorly on questions that require surrounding context.

In the parent-child approach we're studying, child chunks are embedded and indexed for precise retrieval. Each child stores a reference to its parent, which can be fetched after a child matches.

Parent chunks are generally stored separately for context retrieval. Other architectures can index parent chunks too, but that isn't the design in our example.

Sentence-based, token-based and code-aware splitting

semantic chunking uses embedding-based semantic differences to identify likely topic boundaries.

Sentence-based splitting tries to keep complete sentences together, which can preserve meaning and make retrieved passages easier to interpret.

Token-based splitting
Why characters aren't enough
LLMs and embedding models process text in tokens, not characters.

A token may represent a whole word, part of a word, punctuation, or another text unit. Therefore, 500 characters does not reliably equal 500 tokens.

Token-based splitting is useful when you need to control the approximate token budget for embeddings or the LLM context.

Code-aware splitting


Why code needs a different strategy
Suppose you're building a RAG assistant that answers questions about a Python repository.

A generic character splitter might divide a function in the middle:

Code-aware splitting instead tries to respect programming-language structures such as functions, classes, and logical blocks.

Qus=> You're processing a long technical manual with Markdown headings, paragraphs, and code examples. Which splitting approach would you choose, and how would you handle oversized sections?

ans=> I would split the document by Markdown headings first, use code-aware splitting for Python code blocks, and apply recursive splitting to oversized sections. I would preserve heading and source metadata so retrieved chunks remain interpretable.


q=> How would you design a chunking pipeline?
Imagine you're building a chatbot that answers questions from product manuals, PDF guides, and Python documentation.


1. Load documents

PDFs, Markdown, HTML and source code

2. Preserve structure and metadata

Headings, page numbers, source, table headers

3. Select an appropriate splitter

Markdown, code-aware, recursive or semantic

4. Configure size and overlap

Balance precision, context and token budget

5. Evaluate and refine

Measure retrieval quality and answer correctness

q=> How do you evaluate different configurations?

ans=>
I would establish a baseline chunking configuration, evaluate it on representative queries using retrieval and answer-quality metrics, and then compare alternative chunk sizes, overlap values, and splitting strategies. I would select the configuration that provides the best balance of quality, latency, and cost.
